# UrbanTransit IQ — Data Visualization Notebook
**Phase 3: EDA & Quality Visualization using Matplotlib, Seaborn & Plotly**

This notebook loads the **cleaned CSVs** from `processed_data/` and produces rich visualizations:
- Matplotlib & Seaborn: Static publication-quality charts
- Plotly: Interactive charts

**Run `data_cleaning.ipynb` first before running this notebook.**

In [ ]:
# ─── Cell 1: Import All Visualization Libraries ──────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Global Matplotlib style
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

PROCESSED_DIR = '../processed_data/'
CHARTS_DIR    = '../documentation/charts/'
import os; os.makedirs(CHARTS_DIR, exist_ok=True)

print('All libraries loaded successfully!')

---
## 1. Passengers — Passenger Type Distribution & Registration Trend

In [ ]:
# ─── Cell 2: Load passengers_clean.csv ───────────────────────────────────────
passengers = pd.read_csv(PROCESSED_DIR + 'passengers_clean.csv', parse_dates=['registration_date'])
print('Passengers shape:', passengers.shape)
passengers.head(3)

In [ ]:
# ─── Cell 3: Matplotlib — Passenger Type Bar Chart ───────────────────────────
type_counts = passengers['passenger_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(type_counts.index, type_counts.values,
            color=sns.color_palette('husl', len(type_counts)))
axes[0].set_title('Passenger Count by Type', fontweight='bold')
axes[0].set_xlabel('Passenger Type')
axes[0].set_ylabel('Count')
for i, v in enumerate(type_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(type_counts.values, labels=type_counts.index,
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('husl', len(type_counts)))
axes[1].set_title('Passenger Type Share (%)', fontweight='bold')

plt.tight_layout()
plt.savefig(CHARTS_DIR + 'passengers_type_distribution.png')
plt.show()
print('Chart saved.')

In [ ]:
# ─── Cell 4: Seaborn — Monthly Registration Trend ────────────────────────────
passengers['reg_month'] = passengers['registration_date'].dt.to_period('M').astype(str)
monthly_reg = passengers.groupby('reg_month').size().reset_index(name='registrations')

plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_reg, x='reg_month', y='registrations',
             marker='o', linewidth=2.5, color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title('Monthly Passenger Registrations Over Time', fontweight='bold')
plt.xlabel('Month')
plt.ylabel('New Registrations')
plt.tight_layout()
plt.savefig(CHARTS_DIR + 'passengers_monthly_registrations.png')
plt.show()

In [ ]:
# ─── Cell 5: Plotly — Interactive Passenger Type Pie ─────────────────────────
fig = px.pie(
    names=type_counts.index,
    values=type_counts.values,
    title='Passenger Type Distribution (Interactive)',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    hole=0.35
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

---
## 2. Routes — Stop Count & Route Type Distribution

In [ ]:
# ─── Cell 6: Load routes & route_stops ───────────────────────────────────────
routes     = pd.read_csv(PROCESSED_DIR + 'routes_clean.csv')
route_stops = pd.read_csv(PROCESSED_DIR + 'route_stops_clean.csv')
stops_per_route = route_stops.groupby('route_id')['stop_id'].nunique().reset_index()
stops_per_route.columns = ['route_id', 'num_stops']
print('Routes shape:', routes.shape)
routes.head(3)

In [ ]:
# ─── Cell 7: Seaborn — Stops per Route Distribution ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(stops_per_route['num_stops'], bins=20, kde=True,
             color='coral', ax=axes[0])
axes[0].set_title('Distribution of Stops per Route', fontweight='bold')
axes[0].set_xlabel('Number of Stops')
axes[0].set_ylabel('Frequency')

# Top 15 routes by stop count
top15 = stops_per_route.nlargest(15, 'num_stops')
sns.barplot(data=top15, x='num_stops', y='route_id',
            palette='viridis', ax=axes[1])
axes[1].set_title('Top 15 Routes by Number of Stops', fontweight='bold')
axes[1].set_xlabel('Number of Stops')
axes[1].set_ylabel('Route ID')

plt.tight_layout()
plt.savefig(CHARTS_DIR + 'routes_stops_distribution.png')
plt.show()

In [ ]:
# ─── Cell 8: Plotly — Interactive Stops per Route Bar ────────────────────────
fig = px.bar(
    stops_per_route.sort_values('num_stops', ascending=False).head(30),
    x='route_id', y='num_stops',
    title='Stops per Route (Top 30) — Interactive',
    color='num_stops',
    color_continuous_scale='Viridis',
    labels={'num_stops': 'Number of Stops', 'route_id': 'Route ID'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

---
## 3. Vehicles — Capacity Distribution & Vehicle Type

In [ ]:
# ─── Cell 9: Load vehicles_clean.csv ─────────────────────────────────────────
vehicles = pd.read_csv(PROCESSED_DIR + 'vehicles_clean.csv')
print('Vehicles shape:', vehicles.shape)
vehicles.head(3)

In [ ]:
# ─── Cell 10: Matplotlib + Seaborn — Vehicle Capacity Analysis ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Capacity histogram
sns.histplot(vehicles['capacity'], bins=20, kde=True,
             color='teal', ax=axes[0])
axes[0].set_title('Vehicle Capacity Distribution', fontweight='bold')
axes[0].set_xlabel('Capacity (Passengers)')
axes[0].set_ylabel('Count')
axes[0].axvline(vehicles['capacity'].mean(), color='red',
                linestyle='--', label=f"Mean={vehicles['capacity'].mean():.0f}")
axes[0].legend()

# Vehicle type count (if column exists)
if 'vehicle_type' in vehicles.columns:
    type_counts_v = vehicles['vehicle_type'].value_counts()
    sns.barplot(x=type_counts_v.index, y=type_counts_v.values,
                palette='Set2', ax=axes[1])
    axes[1].set_title('Vehicles by Type', fontweight='bold')
    axes[1].set_xlabel('Vehicle Type')
    axes[1].set_ylabel('Count')
    for i, v in enumerate(type_counts_v.values):
        axes[1].text(i, v + 1, str(v), ha='center', fontsize=9)
else:
    axes[1].text(0.5, 0.5, 'vehicle_type column not found',
                 ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig(CHARTS_DIR + 'vehicles_capacity_distribution.png')
plt.show()

In [ ]:
# ─── Cell 11: Plotly — Box Plot of Vehicle Capacity by Type ──────────────────
if 'vehicle_type' in vehicles.columns:
    fig = px.box(
        vehicles, x='vehicle_type', y='capacity',
        title='Vehicle Capacity by Type (Box Plot)',
        color='vehicle_type',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.show()
else:
    fig = px.histogram(vehicles, x='capacity', nbins=20,
                       title='Vehicle Capacity Distribution (Interactive)')
    fig.show()

---
## 4. Trips — Daily Trip Volume & Status Distribution

In [ ]:
# ─── Cell 12: Load trips_clean.csv ───────────────────────────────────────────
print('Loading trips_clean.csv ...')
trips = pd.read_csv(PROCESSED_DIR + 'trips_clean.csv',
                    parse_dates=['actual_departure', 'actual_arrival'],
                    low_memory=False)
print('Trips shape:', trips.shape)
trips.head(3)

In [ ]:
# ─── Cell 13: Seaborn — Daily Trip Volume Heatmap ────────────────────────────
trips['trip_date']    = trips['actual_departure'].dt.date
trips['trip_month']   = trips['actual_departure'].dt.month
trips['trip_weekday'] = trips['actual_departure'].dt.day_name()

# Trips by Month & Weekday heatmap
pivot = trips.groupby(['trip_month', 'trip_weekday']).size().unstack(fill_value=0)
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = pivot[[c for c in weekday_order if c in pivot.columns]]

plt.figure(figsize=(14, 6))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='d', linewidths=0.5)
plt.title('Trip Volume Heatmap — Month vs Weekday', fontweight='bold')
plt.xlabel('Day of Week')
plt.ylabel('Month')
plt.tight_layout()
plt.savefig(CHARTS_DIR + 'trips_heatmap_month_weekday.png')
plt.show()

In [ ]:
# ─── Cell 14: Matplotlib — Trip Status / Cancellation Bar ────────────────────
if 'status' in trips.columns:
    status_counts = trips['status'].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['#2ecc71', '#e74c3c', '#f39c12', '#3498db']

    axes[0].bar(status_counts.index, status_counts.values, color=colors[:len(status_counts)])
    axes[0].set_title('Trip Status Distribution', fontweight='bold')
    axes[0].set_xlabel('Status')
    axes[0].set_ylabel('Count')
    for i, v in enumerate(status_counts.values):
        axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

    axes[1].pie(status_counts.values, labels=status_counts.index,
                autopct='%1.1f%%', colors=colors[:len(status_counts)], startangle=140)
    axes[1].set_title('Trip Status Share (%)', fontweight='bold')

    plt.tight_layout()
    plt.savefig(CHARTS_DIR + 'trips_status_distribution.png')
    plt.show()
else:
    print('status column not found in trips.')

In [ ]:
# ─── Cell 15: Plotly — Monthly Trip Volume Interactive Line Chart ─────────────
monthly_trips = trips.groupby('trip_month').size().reset_index(name='trip_count')

fig = px.line(
    monthly_trips, x='trip_month', y='trip_count',
    title='Monthly Trip Volume (Interactive)',
    markers=True,
    labels={'trip_month': 'Month', 'trip_count': 'Number of Trips'},
    color_discrete_sequence=['#e74c3c']
)
fig.update_layout(xaxis=dict(tickmode='linear', tick0=1, dtick=1))
fig.show()

---
## 5. Delays — Distribution, Cause Analysis & Heatmap

In [ ]:
# ─── Cell 16: Load delays_clean.csv ──────────────────────────────────────────
delays = pd.read_csv(PROCESSED_DIR + 'delays_clean.csv')
print('Delays shape:', delays.shape)
delays.head(3)

In [ ]:
# ─── Cell 17: Matplotlib — Delay Minutes Distribution ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(delays['delay_minutes'], bins=40, color='salmon', edgecolor='white')
axes[0].set_title('Delay Minutes Distribution', fontweight='bold')
axes[0].set_xlabel('Delay (minutes)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(delays['delay_minutes'].mean(), color='navy',
                linestyle='--', label=f"Mean={delays['delay_minutes'].mean():.1f}")
axes[0].legend()

# Box plot
axes[1].boxplot(delays['delay_minutes'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='salmon', color='darkred'))
axes[1].set_title('Delay Minutes Box Plot', fontweight='bold')
axes[1].set_ylabel('Delay (minutes)')
axes[1].set_xticks([])

# Delay category pie (if exists)
if 'delay_category' in delays.columns:
    cat_counts = delays['delay_category'].value_counts()
    axes[2].pie(cat_counts.values, labels=cat_counts.index,
                autopct='%1.1f%%', startangle=140,
                colors=sns.color_palette('Set1', len(cat_counts)))
    axes[2].set_title('Delay Category Share', fontweight='bold')
else:
    bins_d = [0, 5, 15, 30, 60, 300]
    labels_d = ['<5 min','5-15','15-30','30-60','>60']
    delays['delay_band'] = pd.cut(delays['delay_minutes'], bins=bins_d, labels=labels_d)
    band_counts = delays['delay_band'].value_counts().sort_index()
    axes[2].pie(band_counts.values, labels=band_counts.index,
                autopct='%1.1f%%', startangle=140,
                colors=sns.color_palette('Reds', len(band_counts)))
    axes[2].set_title('Delay Severity Bands', fontweight='bold')

plt.tight_layout()
plt.savefig(CHARTS_DIR + 'delays_distribution_analysis.png')
plt.show()

In [ ]:
# ─── Cell 18: Seaborn — Delay Cause Breakdown ────────────────────────────────
if 'cause' in delays.columns:
    cause_avg = delays.groupby('cause')['delay_minutes'].mean().sort_values(ascending=False)

    plt.figure(figsize=(12, 6))
    sns.barplot(x=cause_avg.values, y=cause_avg.index, palette='magma')
    plt.title('Average Delay by Cause', fontweight='bold')
    plt.xlabel('Average Delay (minutes)')
    plt.ylabel('Delay Cause')
    plt.tight_layout()
    plt.savefig(CHARTS_DIR + 'delays_by_cause.png')
    plt.show()
else:
    print('cause column not found in delays — skipping cause breakdown.')

In [ ]:
# ─── Cell 19: Plotly — Interactive Delay Histogram ───────────────────────────
fig = px.histogram(
    delays, x='delay_minutes', nbins=50,
    title='Delay Minutes Distribution (Interactive)',
    color_discrete_sequence=['#e74c3c'],
    labels={'delay_minutes': 'Delay (minutes)'}
)
fig.add_vline(x=delays['delay_minutes'].mean(), line_dash='dash',
              line_color='navy', annotation_text='Mean')
fig.show()

---
## 6. Tickets — Fare Distribution & Ticket Type

In [ ]:
# ─── Cell 20: Load tickets_clean.csv (sample for speed) ─────────────────────
print('Loading tickets_clean.csv (sampling 200K rows for viz) ...')
tickets = pd.read_csv(PROCESSED_DIR + 'tickets_clean.csv',
                      parse_dates=['purchase_time'] if 'purchase_time' in
                      pd.read_csv(PROCESSED_DIR + 'tickets_clean.csv', nrows=1).columns
                      else [],
                      low_memory=False)
# Sample for faster plotting if large
tickets_sample = tickets.sample(min(200_000, len(tickets)), random_state=42)
print('Sample shape:', tickets_sample.shape)

In [ ]:
# ─── Cell 21: Matplotlib — Fare Distribution ─────────────────────────────────
if 'fare_amount' in tickets_sample.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # KDE plot
    sns.kdeplot(tickets_sample['fare_amount'].dropna(),
                fill=True, color='mediumseagreen', ax=axes[0])
    axes[0].set_title('Fare Amount Density (KDE)', fontweight='bold')
    axes[0].set_xlabel('Fare Amount')
    axes[0].set_ylabel('Density')

    # Ticket type if column exists
    if 'ticket_type' in tickets_sample.columns:
        tt_counts = tickets_sample['ticket_type'].value_counts()
        sns.barplot(x=tt_counts.index, y=tt_counts.values,
                    palette='Set3', ax=axes[1])
        axes[1].set_title('Tickets by Type', fontweight='bold')
        axes[1].set_xlabel('Ticket Type')
        axes[1].set_ylabel('Count')
        axes[1].tick_params(axis='x', rotation=30)
    else:
        sns.histplot(tickets_sample['fare_amount'].dropna(),
                     bins=30, color='mediumseagreen', ax=axes[1])
        axes[1].set_title('Fare Amount Histogram', fontweight='bold')

    plt.tight_layout()
    plt.savefig(CHARTS_DIR + 'tickets_fare_distribution.png')
    plt.show()

In [ ]:
# ─── Cell 22: Plotly — Hourly Ticket Purchase Volume ─────────────────────────
if 'purchase_time' in tickets_sample.columns and tickets_sample['purchase_time'].notna().any():
    tickets_sample['purchase_hour'] = pd.to_datetime(tickets_sample['purchase_time'], errors='coerce').dt.hour
    hourly = tickets_sample.groupby('purchase_hour').size().reset_index(name='tickets_sold')

    fig = px.bar(
        hourly, x='purchase_hour', y='tickets_sold',
        title='Ticket Purchases by Hour of Day (Interactive)',
        labels={'purchase_hour': 'Hour', 'tickets_sold': 'Tickets Sold'},
        color='tickets_sold',
        color_continuous_scale='Greens'
    )
    fig.show()

---
## 7. Passenger Counts — Occupancy & Peak Hour Analysis

In [ ]:
# ─── Cell 23: Load passenger_counts_clean.csv (sample) ───────────────────────
print('Loading passenger_counts_clean.csv (sampling 100K rows) ...')
pax_counts = pd.read_csv(PROCESSED_DIR + 'passenger_counts_clean.csv', low_memory=False)
pax_sample = pax_counts.sample(min(100_000, len(pax_counts)), random_state=42)
print('Sample shape:', pax_sample.shape)

In [ ]:
# ─── Cell 24: Seaborn — Occupancy % Distribution ─────────────────────────────
if 'occupancy_pct' in pax_sample.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Violin plot
    sns.violinplot(y=pax_sample['occupancy_pct'].dropna(),
                   color='mediumpurple', ax=axes[0], inner='box')
    axes[0].set_title('Occupancy % Violin Plot', fontweight='bold')
    axes[0].set_ylabel('Occupancy %')
    axes[0].axhline(80, color='red', linestyle='--', label='80% threshold')
    axes[0].legend()

    # Overcrowding flag pie
    overcrowded_count    = (pax_sample['occupancy_pct'] > 100).sum()
    high_risk_count      = ((pax_sample['occupancy_pct'] > 80) & (pax_sample['occupancy_pct'] <= 100)).sum()
    normal_count         = (pax_sample['occupancy_pct'] <= 80).sum()
    labels_occ = ['Normal (<80%)', 'High Risk (80-100%)', 'Overcrowded (>100%)']
    values_occ = [normal_count, high_risk_count, overcrowded_count]
    axes[1].pie(values_occ, labels=labels_occ, autopct='%1.1f%%',
                colors=['#2ecc71', '#f39c12', '#e74c3c'], startangle=140)
    axes[1].set_title('Occupancy Risk Categorization', fontweight='bold')

    plt.tight_layout()
    plt.savefig(CHARTS_DIR + 'occupancy_distribution.png')
    plt.show()

In [ ]:
# ─── Cell 25: Plotly — Boardings vs Alightings Scatter ───────────────────────
if 'boarding_count' in pax_sample.columns and 'alighting_count' in pax_sample.columns:
    scatter_sample = pax_sample.sample(min(5000, len(pax_sample)), random_state=42)
    fig = px.scatter(
        scatter_sample,
        x='boarding_count', y='alighting_count',
        title='Boarding vs Alighting Count per Stop (Interactive)',
        opacity=0.5,
        color='occupancy_pct' if 'occupancy_pct' in scatter_sample.columns else None,
        color_continuous_scale='RdYlGn_r',
        labels={'boarding_count': 'Boardings', 'alighting_count': 'Alightings'}
    )
    fig.show()

---
## 8. GPS Events — Speed Distribution & Vehicle Movement

In [ ]:
# ─── Cell 26: Load gps_events_clean.csv (sample) ─────────────────────────────
print('Loading gps_events_clean.csv (sampling 50K rows) ...')
gps = pd.read_csv(PROCESSED_DIR + 'gps_events_clean.csv',
                  parse_dates=['timestamp'] if 'timestamp' in
                  pd.read_csv(PROCESSED_DIR + 'gps_events_clean.csv', nrows=1).columns
                  else [],
                  low_memory=False)
gps_sample = gps.sample(min(50_000, len(gps)), random_state=42)
print('Sample shape:', gps_sample.shape)

In [ ]:
# ─── Cell 27: Seaborn — Speed Distribution ───────────────────────────────────
if 'speed_kmh' in gps_sample.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.histplot(gps_sample['speed_kmh'].dropna(), bins=40,
                 kde=True, color='dodgerblue', ax=axes[0])
    axes[0].set_title('GPS Vehicle Speed Distribution', fontweight='bold')
    axes[0].set_xlabel('Speed (km/h)')
    axes[0].set_ylabel('Frequency')
    axes[0].axvline(gps_sample['speed_kmh'].mean(), color='red',
                    linestyle='--', label=f"Mean={gps_sample['speed_kmh'].mean():.1f}")
    axes[0].legend()

    # Speed by hour
    if 'timestamp' in gps_sample.columns:
        gps_sample['hour'] = pd.to_datetime(gps_sample['timestamp'], errors='coerce').dt.hour
        hourly_speed = gps_sample.groupby('hour')['speed_kmh'].mean().reset_index()
        sns.lineplot(data=hourly_speed, x='hour', y='speed_kmh',
                     marker='o', color='dodgerblue', ax=axes[1])
        axes[1].set_title('Average Vehicle Speed by Hour', fontweight='bold')
        axes[1].set_xlabel('Hour of Day')
        axes[1].set_ylabel('Avg Speed (km/h)')
    else:
        sns.boxplot(y=gps_sample['speed_kmh'].dropna(), ax=axes[1], color='dodgerblue')
        axes[1].set_title('Speed Box Plot', fontweight='bold')

    plt.tight_layout()
    plt.savefig(CHARTS_DIR + 'gps_speed_distribution.png')
    plt.show()

In [ ]:
# ─── Cell 28: Plotly — Vehicle Location Scatter Map ──────────────────────────
if 'latitude' in gps_sample.columns and 'longitude' in gps_sample.columns:
    map_sample = gps_sample.dropna(subset=['latitude', 'longitude']).head(5000)
    fig = px.scatter_mapbox(
        map_sample,
        lat='latitude', lon='longitude',
        color='speed_kmh' if 'speed_kmh' in map_sample.columns else None,
        color_continuous_scale='RdYlGn',
        zoom=11, height=500,
        title='Vehicle GPS Locations (Interactive Map)',
        mapbox_style='open-street-map'
    )
    fig.show()

---
## 9. Data Quality Summary Dashboard

In [ ]:
# ─── Cell 29: Load Cleaning Audit Log & Visualize ────────────────────────────
audit_df = pd.read_csv(PROCESSED_DIR + 'cleaning_audit_log.csv')
print('Total issues logged:', len(audit_df))
audit_df

In [ ]:
# ─── Cell 30: Matplotlib — Issues per Table Bar Chart ────────────────────────
issues_per_table = audit_df.groupby('table')['original_count'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = sns.color_palette('tab10', len(issues_per_table))
axes[0].barh(issues_per_table.index, issues_per_table.values, color=colors)
axes[0].set_title('Total Issues Detected per Table', fontweight='bold')
axes[0].set_xlabel('Number of Issues')
axes[0].set_ylabel('Table')
for i, v in enumerate(issues_per_table.values):
    axes[0].text(v + 10, i, f'{v:,}', va='center', fontsize=9)

# Issues by type
issues_by_type = audit_df.groupby('issue_type')['original_count'].sum().sort_values(ascending=False)
axes[1].bar(issues_by_type.index, issues_by_type.values,
            color=sns.color_palette('Set2', len(issues_by_type)))
axes[1].set_title('Issues by Type', fontweight='bold')
axes[1].set_xlabel('Issue Type')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.savefig(CHARTS_DIR + 'data_quality_summary.png')
plt.show()

In [ ]:
# ─── Cell 31: Plotly — Interactive Data Quality Dashboard ────────────────────
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Issues per Table', 'Issues by Type'],
                    specs=[[{'type': 'bar'}, {'type': 'pie'}]])

fig.add_trace(
    go.Bar(x=issues_per_table.values, y=issues_per_table.index,
           orientation='h', marker_color='steelblue',
           name='Issues per Table'),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=issues_by_type.index, values=issues_by_type.values,
           name='Issue Types'),
    row=1, col=2
)

fig.update_layout(title_text='UrbanTransit IQ — Data Quality Summary Dashboard',
                  height=500, showlegend=False)
fig.show()
print('\nAll visualizations complete!')